In [0]:
"""
==============================================================
Module: Superstore bronze layer ETL pipeline orchestrator

Purpose:
    This module orchestrates the ETL pipeline for processing data from the raw data sources into the Bronze layer.
    It is responsible for performing incremental data ingestion, applying schema transformations, handling entity-specific
    transformations, and collecting performance metrics for each entity processed.

Key Features:
1. Incremental Data Ingestion:
    - Supports efficient incremental data ingestion from raw source files into the Bronze layer.
    - Handles schema validation, including renaming columns based on configuration and applying metadata transformations.

2. Entity-Specific Processing:
    - Supports entity-specific processing by leveraging configurable transformation logic for different data entities.
    - Ensures that each entity is processed in isolation, with the option to perform additional transformations before ingestion.

3. Schema Management and Configuration:
    - Schema is dynamically loaded based on configuration files to adapt to different environments (e.g., dev, prod).
    - Configurable column renaming and metadata column handling to ensure consistency across environments.

4. Parallel Processing and Partitioning:
    - Utilizes Spark’s parallel processing capabilities for handling large datasets, ensuring efficient ingestion and transformation.
    - Dynamically determines partitioning strategies based on dataset size for optimal performance.

5. Error Handling and Logging:
    - Comprehensive error handling to log and manage any issues encountered during the ingestion process.
    - Detailed logging is available for each step of the pipeline, including successes, failures, and metrics collection.

6. Performance Metrics and Monitoring:
    - Collects detailed metrics for every entity processed, including rows read, rows ingested, throughput, and processing time.
    - Metrics are stored in a separate table for monitoring the pipeline’s performance over time.

7. Production-Ready and Scalable:
    - Designed for production-scale data processing and capable of running efficiently on Databricks serverless clusters.
    - Can be adapted for different environments using environment-specific configurations and paths.

Best Practices / Notes:
- Ensure the `superstore_bronze_config.yaml` file is correctly configured, including file paths and transformation rules.
- Review and monitor the metrics table to track pipeline performance and identify any potential bottlenecks or issues.
- Regularly check the quarantine and audit tables for any data anomalies or issues that may require intervention.
- Be mindful of partitioning strategies to optimize performance, especially for large datasets.
- Use proper exception handling and logging to ensure robust and traceable pipeline execution.

==============================================================
"""

# =========================================================
# SPARK SESSION INITIALIZATION
# =========================================================

from pyspark.sql import SparkSession

# Create or reuse existing Spark session
spark = SparkSession.builder.getOrCreate()

# Set consistent timezone for all timestamp operations (important for ETL consistency)
spark.conf.set("spark.sql.session.timeZone", "UTC")


# =========================================================
# PIPELINE ENVIRONMENT VARIABLES (FROM DATABRICKS JOB CONTEXT)
# =========================================================
import os, sys

# Fetch runtime parameters from upstream initialization task
env = dbutils.jobs.taskValues.get(
    taskKey="superstore_pipeline_master_run_id_init",
    key="SUPERSTORE_ENV"
)

pipeline_name = dbutils.jobs.taskValues.get(
    taskKey="superstore_pipeline_master_run_id_init",
    key="SUPERSTORE_PIPELINE_NAME"
)

pipeline_version = dbutils.jobs.taskValues.get(
    taskKey="superstore_pipeline_master_run_id_init",
    key="SUPERSTORE_PIPELINE_VERSION"
)

# Export environment variables for downstream modules
os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version


# =========================================================
# Add Shared Code Paths
# =========================================================
# this notebook lives at <bundle_root>/superstore_orchestrator/layer_orchestrator/<name>
NOTEBOOK_DIR = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)
BUNDLE_ROOT = "/Workspace" + os.path.dirname(os.path.dirname(NOTEBOOK_DIR))
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_shared_utilities")
sys.path.append(f"{BUNDLE_ROOT}/src/superstore_bronze")   # Add Bronze-specific ingestion modules



# =========================================================
# CORE PLATFORM IMPORTS
# =========================================================
from superstore_logger import get_superstore_logger, log_event
from superstore_orchestrator_execution_context import get_execution_context
from superstore_platform_constants import BRONZE_LAYER
from superstore_platform_config import (
    get_catalog,
    get_bronze_schema,
    get_metrics_schema,
    table
)
from superstore_backfill_utils import (
    get_backfill_config,
    reads_from_source,
    validate_bronze_impact
)


# =========================================================
# EXECUTION CONTEXT (RUN IDENTIFIERS)
# =========================================================
# Fetch master and layer run IDs for traceability across pipeline layers
execution_context = get_execution_context(dbutils, allow_standalone=True)

master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]


# =========================================================
# LOGGER INITIALIZATION
# =========================================================
# Dedicated logger for Bronze orchestrator layer
logger_bronze_orchestrator = get_superstore_logger(
    "superstore_bronze_layer_ETL_orchestrator"
)


# =========================================================
# YAML CONFIG LOADING
# =========================================================
import traceback
import yaml

config_path = f"{BUNDLE_ROOT}/configs/superstore_bronze_config/superstore_bronze_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# =========================================================
# ENVIRONMENT-SPECIFIC PATH RESOLUTION
# =========================================================
# Dynamically resolve storage locations based on environment (dev/qa/prod)
raw_source_file_path = config['env_paths'][env]['raw_source_file_path']
schema_location = config['env_paths'][env]['schema_location']
checkpoint_location = config['env_paths'][env]['checkpoint_location']


# =========================================================
# MODULE IMPORTS (DYNAMIC RELOAD FOR DEV FLEXIBILITY)
# =========================================================
from pyspark.sql.functions import col
from pyspark.sql import DataFrame

import importlib
import time
from datetime import datetime
import pytz
import yaml

# Reload ingestion module (useful during active development)
import bronze_ingest_superstore_module_01
importlib.reload(bronze_ingest_superstore_module_01)
from bronze_ingest_superstore_module_01 import bronze_ingest_incremental

# Reload entity module (incremental append + metrics)
import bronze_entity_superstore_module_02
importlib.reload(bronze_entity_superstore_module_02)
from superstore_schema_drift import (
    is_schema_evolution_error,
    missing_columns_message,
    record_schema_drift,
)

from bronze_entity_superstore_module_02 import (
    get_last_processed_ts,
    bronze_entity_incremental_append,
    collect_entity_metrics
)

# Table layout: each table converges to the CLUSTER BY keys in its config.
from superstore_liquid_clustering import converge_liquid_clustering

# -----------------------------
# Load YAML Bronze Config
# -----------------------------
COLUMN_RENAME_MAP = config["column_rename_map"]
BRONZE_MAIN = config["bronze_main"]
BRONZE_ENTITIES = config["bronze_entities"]

# Source columns knowingly not carried into any entity table. .get() rather than
# [] because a config predating schema-drift detection has no such key, and a
# KeyError here would fail Bronze over a monitoring feature.
IGNORED_SOURCE_COLUMNS = config.get("ignored_source_columns", [])

# -----------------------------
# Log layer start
# -----------------------------
log_event(
    logger_bronze_orchestrator,
    "INFO",
    "Initiating Superstore Bronze layer ETL orchestrator",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=BRONZE_LAYER
)

# -----------------------------
# Run mode
# -----------------------------
# Resolves run_mode / start_date / end_date / dry_run into one validated
# config. Raises on an unknown mode rather than defaulting to incremental:
# a typo must fail the run, not quietly perform a different operation.
backfill_config = get_backfill_config(dbutils, allow_full_refresh=False)
run_mode = backfill_config["mode"]

# -----------------------------
# Replay and full_refresh skip Bronze
# -----------------------------
# Both re-derive Silver and Gold from the Bronze data already held, so there
# is nothing for this task to do. Exiting cleanly (rather than failing) keeps
# the downstream tasks running: they depend on this task succeeding, not on it
# writing. Re-acquiring from source for a replay is wasted work, and once the
# source has aged its files out it is not possible at all.
if not reads_from_source(backfill_config):
    log_event(
        logger_bronze_orchestrator,
        "INFO",
        f"Bronze skipped: run_mode='{run_mode}' re-derives from existing Bronze data",
        run_mode=run_mode,
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=BRONZE_LAYER
    )
    dbutils.notebook.exit(
        f"SKIPPED_BRONZE run_mode={run_mode} (re-deriving downstream from existing Bronze)"
    )

# -----------------------------
# Dry run
# -----------------------------
# Bronze's impact is measured in files, not rows — there is no source table to
# count. Report what Auto Loader would read, then exit before any write.
if backfill_config["dry_run"]:
    impact = validate_bronze_impact(
        dbutils,
        spark,
        raw_source_file_path,
        table(get_bronze_schema(), BRONZE_MAIN["table_name"]),
        backfill_config
    )
    log_event(
        logger_bronze_orchestrator,
        "INFO",
        f"Dry-run impact for Bronze landing path {raw_source_file_path}",
        run_mode=run_mode,
        impact=impact,
        master_run_id=master_run_id,
        layer_run_id=layer_run_id,
        layer=BRONZE_LAYER
    )
    dbutils.notebook.exit(
        f"DRY_RUN_COMPLETED run_mode={run_mode} layer=bronze (no data written)"
    )


# -----------------------------
# Bronze Layer Runner
# -----------------------------
def run_bronze_layer(spark, master_run_id: str, raw_source_file_path:str, schema_location:str, checkpoint_location:str):

    # Use platform config to resolve catalog and schema
    bronze_raw_table = table(get_bronze_schema(), BRONZE_MAIN["table_name"])

    pipeline_start = time.time()

    # -----------------------------
    # 1. Bronze Ingest (Incremental)
    # -----------------------------
    # Auto Loader runs schemaEvolutionMode=addNewColumns, which FAILS the stream
    # on first sight of an unknown column, records the new schema, and expects a
    # restart. Databricks additionally retries this error class by itself, with
    # nothing configured -- so a new source column previously produced a red
    # Bronze task that silently healed on the platform's own retry, leaving a
    # run history saying only "failed, then didn't".
    #
    # Measured 2026-08-17 in dev and integration_test: attempt 0 FAILED with
    # UNKNOWN_FIELD_EXCEPTION, attempt 1 SUCCESS, column added, 0 rescued.
    #
    # Restarting here once, deliberately, and saying why. Bounded to a single
    # extra attempt and gated on is_schema_evolution_error, so genuine faults --
    # a missing file, a permissions error -- still fail on the first attempt
    # rather than looping.
    # superstore_raw is converged to Liquid Clustering BEFORE the stream writes,
    # so the stream never appends into a table that is about to change layout.
    # allow_rewrite=False: Auto Loader streams into this table, so it may only be
    # converted in place (same Delta table id), never rewritten. Its partition
    # column is a DATE, which the in-place conversion supports.
    def _converge_raw_layout():
        converge_liquid_clustering(
            spark, logger_bronze_orchestrator, bronze_raw_table,
            BRONZE_MAIN["cluster_by_columns"], allow_rewrite=False,
            master_run_id=master_run_id, layer_run_id=layer_run_id, layer=BRONZE_LAYER
        )

    _converge_raw_layout()

    for _ingest_attempt in (1, 2):
        try:
            bronze_ingest_incremental(
                spark=spark,
                raw_source_file_path=raw_source_file_path,
                schema_location=schema_location,
                checkpoint_location=checkpoint_location,
                column_rename_map=COLUMN_RENAME_MAP,
                metadata_columns=BRONZE_MAIN["metadata_columns"],
                table_name=bronze_raw_table,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                backfill_config=backfill_config
            )
            break
        except Exception as e:
            # Match against the FULL traceback, not str(e).
            #
            # Measured 2026-08-20: what reaches this handler is Databricks'
            # generic wrapper -- "Some streams terminated before this command
            # could finish!" -- which carries none of the error class. The
            # UNKNOWN_FIELD_EXCEPTION text lives further down the chain, so
            # matching on str(e) returned False and the restart never fired.
            # The bronze task failed and the PLATFORM's retry recovered it,
            # which is exactly the behaviour this was meant to replace.
            _trace = traceback.format_exc()
            if _ingest_attempt == 1 and is_schema_evolution_error(_trace):
                log_event(
                    logger_bronze_orchestrator,
                    "WARNING",
                    f"Auto Loader reported new source column(s). The schema has "
                    f"been recorded; restarting the stream once to pick them up. "
                    f"Underlying error: {e}",
                    schema_evolution_restart=True,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=BRONZE_LAYER
                )
                continue
            log_event(
                logger_bronze_orchestrator,
                "ERROR",
                f"Bronze ingest failed (schema-evolution restart not triggered; "
                f"first 400 chars of trace: {traceback.format_exc()[:400]}): {e}",
                exc_info=True,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=BRONZE_LAYER
            )
            raise

    # Again after ingest: in a fresh environment the stream has only just created
    # the table. A no-op on every other run.
    _converge_raw_layout()

    # -----------------------------
    # 1b. Schema drift check
    # -----------------------------
    # Placed AFTER ingest, so superstore_raw carries the schema that just
    # arrived, and BEFORE the entity loop, which is the whole point of the
    # MISSING case: the loop's select() on an absent column raises an opaque
    # AnalysisException, whereas this names the column and states the
    # consequence while the run can still explain itself.
    #
    # Records only. An undeclared column is still dropped at the entity split --
    # the allowlist is the contract and a human deciding what enters the model
    # is correct. What changes is that the decision stops happening in silence.
    # See docs/SCHEMA_DRIFT.md.
    _drift = None
    try:
        _drift = record_schema_drift(
            spark=spark,
            logger=logger_bronze_orchestrator,
            raw_table=bronze_raw_table,
            drift_table=table(get_metrics_schema(), "schema_drift"),
            bronze_entities=BRONZE_ENTITIES,
            ignored_source_columns=IGNORED_SOURCE_COLUMNS,
            master_run_id=master_run_id,
            layer_run_id=layer_run_id,
            env=env,
        )
    except Exception as e:
        # Deliberately does NOT re-raise, which cuts against this codebase's
        # usual rule -- so the reasoning is written down rather than assumed.
        #
        # Everything else that swallowed an exception here was performing a DATA
        # operation, and a silent failure meant wrong or missing rows. This is
        # an instrument. If it breaks, the load is still correct, and failing
        # the run would make adding a monitor strictly riskier than having none
        # -- which is how teams stop adding them.
        #
        # It is not silent, and that is what makes the trade acceptable: this
        # logs at ERROR, and record_schema_drift writes a RESCUED row on EVERY
        # run, so a run with no row in schema_drift is itself evidence the check
        # did not execute. A broken monitor here is detectable by querying the
        # table it was supposed to write.
        log_event(
            logger_bronze_orchestrator,
            "ERROR",
            f"Schema drift check failed; load continues: {e}",
            exc_info=True,
            master_run_id=master_run_id,
            layer_run_id=layer_run_id,
            layer=BRONZE_LAYER
        )

    # A declared column that stopped arriving is a DATA condition, not a broken
    # instrument, so it is raised OUTSIDE the try/except above. Inside it, the
    # same handler that deliberately protects the pipeline from a failing
    # monitor would swallow a genuinely fatal drift.
    #
    # Raised here rather than left to the entity split, which selects declared
    # columns by name and surfaces `AnalysisException: cannot resolve segment`
    # several steps after the cause was already known -- detect_drift computes
    # `missing` before this loop runs.
    if _drift and _drift.get("missing"):
        raise RuntimeError(
            missing_columns_message(_drift["missing"], BRONZE_ENTITIES, env)
        )

    # -----------------------------
    # 2. Loop over Bronze Entities
    # -----------------------------
    for entity_name, cfg in BRONZE_ENTITIES.items():

        metadata_columns = cfg.get("META_COLUMNS", [])
        metrics_table = table(get_metrics_schema(), cfg["metrics_table"].split('.')[-1])  # dynamically resolve catalog+schema
        entity_table_name = table(get_bronze_schema(), cfg["table_name"].split('.')[-1])
        source_table = table(get_bronze_schema(), cfg["source_table"].split('.')[-1])
        layer_name = cfg["layer_name"]

        start_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
        start_time_epoch = time.time()
        # run_status = "SUCCESS"
        # notes = ""

        try:
            bronze_df = spark.table(bronze_raw_table).select('*')
            last_processed_ts = get_last_processed_ts(spark, entity_table_name)

            if last_processed_ts:
                incremental_df = bronze_df.filter(col("bronze_ingestion_ts") > last_processed_ts)
            else:
                incremental_df = bronze_df

            has_incremental_rows = incremental_df.limit(1).count()

            if not has_incremental_rows:
                log_event(
                    logger_bronze_orchestrator,
                    "INFO",
                    f"Entity '{entity_name}' has no new rows/files for this layer run; skipping Bronze append.",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=BRONZE_LAYER
                )
                empty_df = spark.createDataFrame([], bronze_df.schema)
                collect_entity_metrics(
                    spark=spark,
                    entity_df=empty_df,
                    metrics_table=metrics_table,
                    load_type= "NO_DATA",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer_name=layer_name,
                    source_table=source_table,
                    entity_name=entity_name,
                    target_table=entity_table_name,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0
                )
            else:
                result = bronze_entity_incremental_append(
                    entity_name=entity_name,
                    entity_cfg=cfg,
                    spark=spark,
                    metadata_columns=metadata_columns,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id
                )

                # -----------------------------
                # Extract outputs
                # -----------------------------
                df_entity = result["entity_df"]
                load_type = result["load_type"]

                end_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
                duration_secs = int(time.time() - start_time_epoch)

                collect_entity_metrics(
                    spark=spark,
                    entity_df=df_entity,
                    metrics_table=metrics_table,
                    load_type=load_type,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer_name=layer_name,
                    source_table=source_table,
                    entity_name=entity_name,
                    target_table=entity_table_name,
                    start_ts=start_ts,
                    end_ts=end_ts,
                    duration_secs=duration_secs
                    # run_status=run_status,
                    # notes=notes
                )

                log_event(
                    logger_bronze_orchestrator,
                    "INFO",
                    f"Completed Bronze entity processing: {entity_name}",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=BRONZE_LAYER
                )

            # After the append, whether or not this run had rows: converges an
            # existing table, and a table the append has just created.
            # allow_rewrite=True: entity tables are partitioned on a TIMESTAMP,
            # which the in-place conversion cannot handle, so they are rewritten
            # once. Safe because they are written and read in batch only -- no
            # stream depends on their Delta table id.
            converge_liquid_clustering(
                spark, logger_bronze_orchestrator, entity_table_name,
                cfg["cluster_by_columns"], allow_rewrite=True,
                master_run_id=master_run_id, layer_run_id=layer_run_id, layer=BRONZE_LAYER
            )

        except Exception as e:
            run_status = "FAILED"
            notes = str(e)
            log_event(
                logger_bronze_orchestrator,
                "ERROR",
                f"Bronze entity pipeline failed for {entity_name}: {e}",
                exc_info=True,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=BRONZE_LAYER
            )
            raise

    # -----------------------------
    # Pipeline End
    # -----------------------------
    pipeline_end = time.time()
    log_event(logger_bronze_orchestrator, 
              "INFO", f"Bronze unified pipeline completed successfully. Total duration: {round(pipeline_end - pipeline_start, 2)}s", 
              master_run_id=master_run_id,
              layer_run_id=layer_run_id,
              layer=BRONZE_LAYER
            )

# -----------------------------
# Call Bronze Layer Inline
# -----------------------------
if __name__ == "__main__":
    run_bronze_layer(spark, master_run_id, raw_source_file_path, schema_location, checkpoint_location)